# ControlDiff Quickstart Guide

This notebook demonstrates basic usage of ControlDiff for material-conditioned image generation.

In [ ]:
import torch
import sys
sys.path.append('../src')

from models import UNet2DConditionModel, AutoencoderKL, DDIMScheduler
from conditioning import MaterialEncoder, compute_brdf_params
from evaluation import visualize_generation

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Load Models

In [ ]:
# Initialize models
unet = UNet2DConditionModel(
    in_channels=4,
    out_channels=4,
    cross_attention_dim=768,
).to(device)

vae = AutoencoderKL(
    latent_channels=4,
).to(device)

scheduler = DDIMScheduler(num_train_timesteps=1000)

# Load pretrained weights (if available)
# unet.load_state_dict(torch.load('checkpoints/model.pt'))

unet.eval()
vae.eval()

print('Models loaded successfully!')

## 2. Material-Conditioned Generation

In [ ]:
# Initialize material encoder
material_encoder = MaterialEncoder(embed_dim=768).to(device)

# Define material properties
# Example: Brushed metal
brdf_params = compute_brdf_params(
    roughness=0.3,
    metallic=0.9,
    base_color=(0.8, 0.8, 0.8),
    specular=0.7,
).unsqueeze(0).to(device)

# Encode material
material_embedding = material_encoder.encode_parameters(brdf_params)

print(f'Material embedding shape: {material_embedding.shape}')

In [ ]:
# Generate image
@torch.no_grad()
def generate(prompt_embedding, num_steps=50):
    scheduler.set_timesteps(num_steps)
    
    # Start from noise
    latents = torch.randn(1, 4, 64, 64, device=device)
    
    # Denoising loop
    for t in scheduler.timesteps:
        timestep = t.to(device).unsqueeze(0)
        
        noise_pred = unet(
            latents,
            timestep,
            encoder_hidden_states=prompt_embedding,
        )
        
        latents = scheduler.step(noise_pred, int(t), latents)
    
    # Decode
    image = vae.decode(latents)
    return image

# Generate
generated_image = generate(material_embedding.unsqueeze(1))

# Visualize
visualize_generation(generated_image, prompts=['Brushed metal material'])

## 3. Experiment with Different Materials

In [ ]:
# Define multiple materials
materials = [
    {'name': 'Polished Metal', 'roughness': 0.1, 'metallic': 1.0, 'color': (0.8, 0.8, 0.8)},
    {'name': 'Rough Plastic', 'roughness': 0.7, 'metallic': 0.0, 'color': (0.2, 0.5, 0.8)},
    {'name': 'Wood', 'roughness': 0.5, 'metallic': 0.0, 'color': (0.6, 0.4, 0.2)},
    {'name': 'Glass', 'roughness': 0.05, 'metallic': 0.1, 'color': (0.9, 0.9, 0.95)},
]

generated_images = []
prompts = []

for mat in materials:
    # Compute BRDF parameters
    params = compute_brdf_params(
        roughness=mat['roughness'],
        metallic=mat['metallic'],
        base_color=mat['color'],
    ).unsqueeze(0).to(device)
    
    # Encode and generate
    embedding = material_encoder.encode_parameters(params)
    image = generate(embedding.unsqueeze(1), num_steps=30)
    
    generated_images.append(image)
    prompts.append(mat['name'])

# Visualize all
all_images = torch.cat(generated_images, dim=0)
visualize_generation(all_images, prompts=prompts, num_images=4, nrow=2)

## 4. Save Generated Images

In [ ]:
from torchvision.utils import save_image
from pathlib import Path

output_dir = Path('../outputs/quickstart')
output_dir.mkdir(parents=True, exist_ok=True)

for i, (image, prompt) in enumerate(zip(generated_images, prompts)):
    filename = f"{prompt.lower().replace(' ', '_')}.png"
    save_image(
        (image + 1) / 2,  # Denormalize
        output_dir / filename
    )
    print(f'Saved: {filename}')